# 00 — Panda Robot Introduction

This notebook introduces a real redundant robotic system before we build our
own models from scratch.

Goals:

- Understand the configuration vector $q$
- Compare two configurations: `qz` and `qr`
- See how joint configuration determines end-effector position
- Introduce the Jacobian $J(q)$
- Interpret its rows and columns
- Connect the Jacobian to rank, nullspace, redundancy, and singularities

In [1]:
import numpy as np
import pandas as pd
import roboticstoolbox as rtb
from IPython.display import display

np.set_printoptions(precision=3, suppress=True)

panda = rtb.models.URDF.Panda()

print("Robot:", panda.name)
print("Number of joints:", panda.n)

Robot: panda
Number of joints: 7


## 1. Configuration $q$

The Panda has 7 revolute joints.

Its configuration is represented by

$$
q =
\begin{bmatrix}
q_1 \\
q_2 \\
q_3 \\
q_4 \\
q_5 \\
q_6 \\
q_7
\end{bmatrix}
$$

Each $q_i$ is a joint angle.

Robotics Toolbox provides several named configurations. Two useful ones are:

- `qz`: zero configuration
- `qr`: ready configuration

The toolbox internally indexes the joints from 0 through 6, while mathematically we will usually call them $q_1,\ldots,q_7$.

In [2]:
configs = pd.DataFrame(
    {
        "qz (rad)": panda.qz,
        "qz (deg)": np.rad2deg(panda.qz),
        "qr (rad)": panda.qr,
        "qr (deg)": np.rad2deg(panda.qr),
    },
    index=[f"Joint {i}" for i in range(1, 8)]
)

configs.round(2)

,qz (rad),qz (deg),qr (rad),qr (deg)
Joint 1,0.0,0.0,0.00,0.00
Joint 2,0.0,0.0,-0.30,-17.19
Joint 3,0.0,0.0,0.00,0.00
Joint 4,0.0,0.0,-2.20,-126.05
Joint 5,0.0,0.0,0.00,0.00
Joint 6,0.0,0.0,2.00,114.59
Joint 7,0.0,0.0,0.79,45.00


## 2. What do these configurations look like?

First, look at the zero configuration $q_z$.

In [3]:
panda.plot(panda.qz, backend="swift")

Swift backend, t = 0.05, scene:
  [0] AssemblyHandle(robot='panda', links=9)
      panda_link0 (1 geometry, 3 collision)
      panda_link1 (1 geometry, 3 collision)
      panda_link2 (1 geometry, 3 collision)
      panda_link3 (1 geometry, 3 collision)
      panda_link4 (1 geometry, 3 collision)
      panda_link5 (1 geometry, 6 collision)
      panda_link6 (1 geometry, 3 collision)
      panda_link7 (1 geometry, 3 collision)
      panda_link8 (3 collision)

Now compare it with the ready configuration $q_r$.

In [4]:
panda.plot(panda.qr, backend="swift")

Swift backend, t = 0.05, scene:
  [0] AssemblyHandle(robot='panda', links=9)
      panda_link0 (1 geometry, 3 collision)
      panda_link1 (1 geometry, 3 collision)
      panda_link2 (1 geometry, 3 collision)
      panda_link3 (1 geometry, 3 collision)
      panda_link4 (1 geometry, 3 collision)
      panda_link5 (1 geometry, 6 collision)
      panda_link6 (1 geometry, 3 collision)
      panda_link7 (1 geometry, 3 collision)
      panda_link8 (3 collision)

### Interactive joint exploration

`teach()` lets us change the components of $q$ directly.

Moving one slider changes one joint angle $q_i$.
Later, the columns of the Jacobian will describe the instantaneous effect
of moving each of these joints.

In [ ]:
panda.teach(panda.qr, backend="swift")

> Note: Closing the Swift interactive viewer may produce a shutdown/close error in Jupyter.  
> The visualization itself still works as expected.

## 3. Forward kinematics

Joint configuration determines the pose of the end effector.

We can write this abstractly as

$$
x = f(q)
$$

where $q$ contains the joint angles and $x$ describes the position and
orientation of the robot's end effector.

This mapping is called **forward kinematics**.

In [6]:
T_z = panda.fkine(panda.qz)
T_r = panda.fkine(panda.qr)

print("End-effector position at qz:")
print(T_z.t)

print("\nEnd-effector position at qr:")
print(T_r.t)

End-effector position at qz:
[ 0.088 -0.     0.823]

End-effector position at qr:
[ 0.484 -0.     0.413]


## 4. The Jacobian

Forward kinematics gives

$$
x = f(q).
$$

The Jacobian describes how small changes in joint configuration affect
the motion of the end effector:

$$
\dot{x} = J(q)\dot{q}.
$$

For the Panda,

$$
J(q) \in \mathbb{R}^{6 \times 7}.
$$

The 7 columns correspond to the 7 joints.

The 6 rows describe end-effector velocity:

$$
\begin{bmatrix}
v_x \\
v_y \\
v_z \\
\omega_x \\
\omega_y \\
\omega_z
\end{bmatrix}.
$$

Therefore, each **column** answers:

> If this joint moves right now, in what direction does the end effector move?

The Jacobian depends on configuration, so generally

$$
J(q_z) \neq J(q_r).
$$

In [7]:
J_z = panda.jacob0(panda.qz)
J_r = panda.jacob0(panda.qr)

row_labels = [
    "vx",
    "vy",
    "vz",
    "ωx",
    "ωy",
    "ωz",
]

column_labels = [
    "q̇1",
    "q̇2",
    "q̇3",
    "q̇4",
    "q̇5",
    "q̇6",
    "q̇7",
]

Jz_table = pd.DataFrame(
    J_z,
    index=row_labels,
    columns=column_labels
)

display(Jz_table.round(3))

,q̇1,q̇2,q̇3,q̇4,q̇5,q̇6,q̇7
vx,0.000,0.490,0.000,-0.174,0.000,0.210,0.0
vy,0.088,-0.000,0.088,0.000,0.088,0.000,0.0
vz,-0.000,-0.088,0.000,0.005,0.000,0.088,0.0
ωx,-0.000,0.000,0.000,0.000,0.000,0.000,0.0
ωy,0.000,1.000,-0.000,-1.000,-0.000,-1.000,-0.0
ωz,1.000,0.000,1.000,0.000,1.000,0.000,-1.0


### Reading one column

Column $j_i$ of the Jacobian describes the end-effector velocity produced
by joint $i$ moving at one radian per second while the other joints remain
stationary.

In [8]:
joint = 1

col = Jz_table.iloc[:, joint - 1].copy()

col[np.abs(col) < 1e-10] = 0

col.round(3)

vx    0.000
vy    0.088
vz    0.000
ωx    0.000
ωy    0.000
ωz    1.000
Name: q̇1, dtype: float64

## 5. Configuration changes the Jacobian

In [9]:
display(
    pd.DataFrame(
        {
            "qz": {
                "rank": np.linalg.matrix_rank(J_z),
                "nullity": panda.n - np.linalg.matrix_rank(J_z),
            },
            "qr": {
                "rank": np.linalg.matrix_rank(J_r),
                "nullity": panda.n - np.linalg.matrix_rank(J_r),
            },
        }
    )
)

,qz,qr
rank,5,6
nullity,2,1


At $q_z$,

$$
\operatorname{rank}(J)=5.
$$

At $q_r$,

$$
\operatorname{rank}(J)=6.
$$

This is important: **the robot itself did not change. Its configuration did.**

At the zero pose, the geometry causes one normally available task-space
direction to be lost. This is a kinematic singularity.

Using rank-nullity,

$$
\operatorname{rank}(J) + \operatorname{nullity}(J) = 7.
$$

Therefore:

$$
q_z:\qquad 5 + 2 = 7
$$

while

$$
q_r:\qquad 6 + 1 = 7.
$$

The ready pose has one redundant joint-motion direction.

The zero pose has two nullspace directions because normal redundancy is
combined with a singular configuration.

## 6. Why this matters

The Jacobian gives us two important linear-algebra objects.

### Column space

$$
\operatorname{Col}(J)
$$

contains the instantaneous end-effector velocities accessible from the
current configuration.

### Nullspace

$$
\ker(J)
$$

contains joint velocities satisfying

$$
J(q)\dot q = 0.
$$

These are internal joint motions that produce no instantaneous change in
the selected end-effector task.

This gives us an important principle for later human models:

$$
\boxed{\text{same task output does not uniquely determine internal motion}}
$$

and also:

$$
\boxed{\text{available movement depends on configuration}}
$$

## 7. Exercise — Does the Jacobian actually predict motion?

The Jacobian is supposed to describe how joint velocity produces instantaneous
end-effector velocity:

$$
\dot{x} = J(q)\dot{q}
$$

We can test this numerically.

Starting from the ready configuration $q_r$, we will:

1. Choose one joint to move.
2. Use the Jacobian to predict the end-effector's translational velocity.
3. Move the joint by a tiny amount.
4. Compute the actual change in end-effector position.
5. Compare the prediction with the observed motion.

For a sufficiently small time step $\Delta t$,

$$
\dot{x}
\approx
\frac{f(q + \Delta t\,\dot{q}) - f(q)}{\Delta t}
$$

If the Jacobian really is the local derivative of the forward-kinematics
mapping, these two calculations should be nearly identical.

### Before running the code

We will move only Joint 1 at

$$
\dot{q}_1 = 1 \text{ rad/s}
$$

while all other joint velocities are zero.

Based on what we learned about Jacobian columns, which part of $J_r$ should
contain the predicted end-effector velocity?

In [10]:
# Start at the ready configuration
q = panda.qr.copy()

# Move only Joint 1 at 1 rad/s
qdot = np.zeros(panda.n)
qdot[0] = 1.0

# Very small time step
dt = 1e-6

# Jacobian at the current configuration
J = panda.jacob0(q)

# Predicted translational velocity from the Jacobian
v_predicted = J[:3, :] @ qdot

# Current end-effector position
p0 = panda.fkine(q).t

# Move the joints forward by a tiny amount
q_next = q + dt * qdot

# New end-effector position
p1 = panda.fkine(q_next).t

# Approximate actual translational velocity
v_actual = (p1 - p0) / dt

comparison = pd.DataFrame(
    {
        "Jacobian prediction": v_predicted,
        "Finite difference": v_actual,
        "Difference": v_actual - v_predicted,
    },
    index=["vx", "vy", "vz"],
)

display(comparison.round(8))

,Jacobian prediction,Finite difference,Difference
vx,0.000000,-2.400000e-07,-2.400000e-07
vy,0.484047,4.840468e-01,-0.000000e+00
vz,0.000000,0.000000e+00,-0.000000e+00


In [11]:
error = np.linalg.norm(v_actual - v_predicted)

print(f"Prediction error: {error:.3e}")

Prediction error: 2.420e-07


### Interpretation

The Jacobian prediction and the finite-difference calculation are nearly
identical.

For Joint 1, the Jacobian predicts approximately

$$
v_y = 0.484047
$$

while directly evaluating the forward kinematics after a tiny joint motion
produces essentially the same velocity.

The very small difference in $v_x$ comes from the fact that the robot's full
kinematics are nonlinear, while the Jacobian is a local linear approximation.

Thus,

$$
\Delta x \approx J(q)\Delta q
$$

for sufficiently small changes in configuration.

This is why linear algebra remains useful even though the full mechanical
system is nonlinear: around any particular configuration, the Jacobian gives
us a local linear model.

The next models will reproduce these ideas with simpler systems whose
kinematics and Jacobians we derive ourselves.